In [1]:
# Cell 1: load libraries and preprocessed UAV data

import numpy as np  # used it for working with arrays and loading .npy files

import pandas as pd  # used it for reading CSV files and working with table-like data

import tensorflow as tf  # used it for deep learning / neural network models

from tensorflow import keras  # keras is a high-level API used to build neural networks more easily


from sklearn.metrics import (
    accuracy_score,  # how many predictions are correct overall

    precision_score,  # predicted positive cases, how many are actually correct

    recall_score,  # actual positive cases, how many the model found

    f1_score,  # balance between precision and recall

    classification_report  # shows precision, recall, f1_score, and support together
)


import json  # used to save or read structured data (dictionary format)

import time  # used to measure execution time


print(f"TensorFlow Version: {tf.__version__}")  
# check which TensorFlow version is installed


# this avoids rewriting the long path multiple times
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"


X_train = np.load(save_path + "X_train.npy")
# load training features; np.load() is used because file is saved as .npy format

X_test = np.load(save_path + "X_test.npy")
# load testing features


y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
# load training labels. read_csv() reads CSV file;
# squeeze() converts DataFrame -> Series (1 column only)

y_test = pd.read_csv(save_path + "y_test.csv").squeeze()
# load testing labels


# load label classes, converts label numbers back to attack names/classes;
# tolist() converts Series -> Python list
le_classes = pd.read_csv(
    save_path + "label_classes.csv"
).squeeze().tolist()


# load feature names, stores all column/feature names used in the dataset
feature_names = pd.read_csv(
    save_path + "feature_names.csv"
).squeeze().tolist()


print(f"X_train: {X_train.shape}")

print(f"X_test: {X_test.shape}")

print(f"Classes: {le_classes}") # print class labels like: normal, attack1, attack2...

print(f"Features: {len(feature_names)}")

print("\nData loaded!")

2026-05-09 19:45:36.901144: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow Version: 2.16.2
X_train: (23171, 37)
X_test: (9931, 37)
Classes: ['DoS attack', 'Replay', 'benign']
Features: 37

Data loaded!


In [2]:
# Cell 2: encode labels and reshape data for CNN

from sklearn.preprocessing import LabelEncoder  
# used to convert text labels into numbers


# STEP 1: ENCODE LABELS

le = LabelEncoder()  
# LabelEncoder converts text labels to numeric labels
# e.g.: 'benign' -> 0 ; 'DoS attack' -> 1 ; 'Replay' -> 2


# fit_transform():
# 1: learn the mapping between text and numbers
# 2: apply the mapping to training labels
# e.g.: normal -> 0 ; attack -> 1
y_train_enc = le.fit_transform(y_train)


# transform() only applies the same mapping;
# it does not relearn new mappings
# test data must use the same label mapping as training data
y_test_enc = le.transform(y_test)


# count how many classes exist
# le.classes_ stores all unique class names
# len() counts total number of classes
num_classes = len(le.classes_)


# list() converts NumPy object into Python list
# for easier viewing
print(f"Classes: {list(le.classes_)}")

print(f"Number of classes: {num_classes}")


# STEP 2: RESHAPE DATA FOR 1D-CNN

# current shape (2D):
# (samples, 37) = (1000 rows, 37 features)

# CNN expects 3D input:
# (samples, 37, 1)
# = (1000 rows, 37 features, 1 channel)

# similar idea:
# grayscale image = 1 channel
# RGB image = 3 channels

X_train_cnn = X_train.reshape(
    X_train.shape[0],  # number of rows (samples)
    X_train.shape[1],  # number of columns (features)
    1
)


# reshape testing data using the same format
X_test_cnn = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)


print(f"\nX_train reshaped: {X_train_cnn.shape}")

print(f"X_test reshaped: {X_test_cnn.shape}")

print("Reshape complete!")

Classes: [0, 1, 2]
Number of classes: 3

X_train reshaped: (23171, 37, 1)
X_test reshaped: (9931, 37, 1)
Reshape complete!


In [3]:
# Cell 3: build 1D-CNN model architecture

print("Building 1D-CNN model...")


# Sequential() = build model layer by layer
model = keras.Sequential([

    # input layer:
    # tells CNN what input shape to expect
    # shape = (37,1)
    # 37 = number of features
    # 1 = one channel
    keras.layers.Input(
        shape=(X_train_cnn.shape[1], 1)
    ),


    # Convolution layer 1
    keras.layers.Conv1D(
        filters=64,  # CNN learns 64 different pattern detectors

        kernel_size=3,  # looks at 3 features at a time

        activation='relu',  # removes negative values, helps learn non-linear patterns

        padding='same'  # keeps output size same as input
    ),


    # MaxPooling1D:
    # keeps strongest signals
    # reduces data size
    keras.layers.MaxPooling1D(
        pool_size=2  # reduces size by half
    ),


    # Convolution layer 2
    # second Conv1D, learns deeper / more complex patterns
    keras.layers.Conv1D(
        filters=128,  # more pattern detectors

        kernel_size=3,

        activation='relu',

        padding='same'
    ),


    # second pooling layer
    # reduces feature size again
    keras.layers.MaxPooling1D(
        pool_size=2
    ),


    # Flatten:
    # converts multi-dimensional data into 1D
    # needed before Dense layers
    keras.layers.Flatten(),


    # Dense layer:
    # fully connected neural network layer
    keras.layers.Dense(
        128,  # neurons for learning higher-level patterns
        activation='relu'
    ),


    # Dropout:
    # randomly turns off 30% neurons
    # helps prevent overfitting
    # makes model generalize better
    keras.layers.Dropout(
        0.3
    ),


    # output layer:
    # one neuron for each class

    # softmax:
    # converts outputs into probabilities
    # all probabilities sum to 1
    keras.layers.Dense(
        num_classes,
        activation='softmax'
    )
])


# COMPILE MODEL
model.compile(

    # optimizer:
    # controls how model updates weights
    # adam = popular optimizer, fast and adaptive
    optimizer='adam',

    # loss function:
    # used for multi-class classification

    # sparse = labels are integers (0,1,2...)
    loss='sparse_categorical_crossentropy',

    # evaluation metric
    # accuracy = how many predictions are correct
    metrics=['accuracy']
)


# show model architecture summary
# displays layers, shapes, and parameters
model.summary()

print("\nModel built!")

Building 1D-CNN model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 37, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 18, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 9, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 172,931 (675.51 KB)

 Trainable params: 172,931 (675.51 KB)

 Non-trainable params: 0 (0.00 B)


Model built!


In [4]:
# Cell 4: Train 1D-CNN model
print("Training 1D-CNN...")
print("This will take a few minutes...")

start_time = time.time()

history = model.fit(
    X_train_cnn, y_train_enc, # training data and labels
    epochs=5,                 # number of full passes through data(trains through all training data 5 times)
    batch_size=512,           # samples per weight update (processes 512 samples at once before updating weights)
    validation_split=0.1,     # use 10% of training data for validation(to monitor overfitting)
    verbose=1                 # show process bar
)
end_time = time.time()
cnn_train_time = round(end_time - start_time,2)

print(f"\nTraining complete!")
print(f"Training time: {cnn_train_time} seconds")

Training 1D-CNN...
This will take a few minutes...
Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.5760 - loss: 0.8512 - val_accuracy: 0.8136 - val_loss: 0.4691
Epoch 2/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - accuracy: 0.8297 - loss: 0.4255 - val_accuracy: 0.8848 - val_loss: 0.2875
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.8857 - loss: 0.2772 - val_accuracy: 0.8934 - val_loss: 0.2438
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.9101 - loss: 0.2269 - val_accuracy: 0.9206 - val_loss: 0.1933
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.9279 - loss: 0.1801 - val_accuracy: 0.9556 - val_loss: 0.1582

Training complete!
Training time: 22.16 seconds


In [10]:
# Cell 5: Evaluate CNN model on test data

print("Evaluating 1D-CNN...")

start_time = time.time()

# predict() returns probability for each class
# Example: [[0.05, 0.90, 0.05]] = 90% chance of class 1
y_pred_proba = model.predict(X_test_cnn, batch_size=512, verbose=1)

# argmax() picks class with highest probability
# axis=1 = pick maximum across columns (classes)
y_pred_enc = np.argmax(y_pred_proba, axis=1)

end_time = time.time()
cnn_pred_time = round(end_time - start_time, 2)

# inverse_transform() converts numbers back to class names
# Example: 0 → 'DoS attack', 1 → 'Replay', 2 → 'benign'
y_pred_labels = le.inverse_transform(y_pred_enc)

# Calculate metrics
acc = accuracy_score(y_test, y_pred_labels)
p   = precision_score(y_test, y_pred_labels, average='weighted')
r   = recall_score(y_test, y_pred_labels, average='weighted')
f1  = f1_score(y_test, y_pred_labels, average='weighted')

print(f"\n=== 1D-CNN Results (UAV Dataset) ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Prediction time: {cnn_pred_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred_labels,
      target_names=le_classes))

Evaluating 1D-CNN...
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

=== 1D-CNN Results (UAV Dataset) ===
Accuracy:  0.9606 (96.06%)
Precision: 0.9613 (96.13%)
Recall:    0.9606 (96.06%)
F1-Score:  0.9606 (96.06%)
Prediction time: 0.94s

Detailed Report:
              precision    recall  f1-score   support

  DoS attack       0.99      0.94      0.96      3501
      Replay       0.94      0.96      0.95      3602
      benign       0.96      0.99      0.98      2828

    accuracy                           0.96      9931
   macro avg       0.96      0.96      0.96      9931
weighted avg       0.96      0.96      0.96      9931



In [11]:
# Cell 6: Save CNN model and results

# Save model in keras format
# .keras saves: architecture + weights + training config
model.save(save_path + "cnn_model.keras")
print("CNN model saved!")

# Save results as JSON for benchmark comparison
cnn_results = {
    "model": "1D-CNN",
    "dataset": "UAV-Cyber-Physical",
    "accuracy":  round(float(acc), 4),
    "precision": round(float(p), 4),
    "recall":    round(float(r), 4),
    "f1_score":  round(float(f1), 4),
    "training_time": cnn_train_time,
    "prediction_time": cnn_pred_time,
    "epochs": 5,
    "batch_size": 512,
    "classes": le_classes
}

with open(save_path + "cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=4)
print("CNN results JSON saved!")

print(f"\nSummary:")
print(f"  Accuracy:   {acc*100:.2f}%")
print(f"  F1-Score:   {f1*100:.2f}%")
print(f"  Train time: {cnn_train_time}s")

CNN model saved!
CNN results JSON saved!

Summary:
  Accuracy:   96.06%
  F1-Score:   96.06%
  Train time: 22.16s
